# ADS Homework 3 - Part 3: RNN (PyTorch)


This notebook follows `.cursor/rules/task_description_hw3.mdc` and the plan in `hw3_plan.md`.


**Dataset (Kaggle path):**
- **Jena Climate**: `/kaggle/input/jena-climate`

We use PyTorch for sequence modeling with Vanilla RNN, LSTM, and GRU experiments.

**Author:** [Your Name]


In [ ]:
# Core imports and setup
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 1) Dataset Path
Confirm this path matches the Kaggle mount.

- **Jena Climate**: `/kaggle/input/jena-climate`


In [ ]:
# Update if your Kaggle path differs
JENA_ROOT = Path("/kaggle/input/jena-climate")

print("Jena Climate exists:", JENA_ROOT.exists())


# Part 3: RNN on Jena Climate (Time Series)

**Task:** Forecast `T (degC)` (next step or future window).

**Preprocessing:** Sliding windows (e.g., input 24/72h -> predict next), normalize features.

**Models:** Vanilla RNN, LSTM, GRU.

**Experiments:** Sequence length, hidden size, one vs multiple recurrent layers, bidirectional vs unidirectional, dropout between recurrent layers.

**Metrics:** MAE, MSE, loss curves.


In [ ]:
def load_jena_data(root_path):
    csvs = list(root_path.glob("*.csv"))
    if not csvs:
        return None
    df = pd.read_csv(csvs[0])
    if "T (degC)" in df.columns:
        series = df["T (degC)"].values.astype(np.float32)
    else:
        series = df.iloc[:, 1].values.astype(np.float32)
    return series


def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)


train_data = None
val_data = None

if JENA_ROOT.exists():
    temperature_data = load_jena_data(JENA_ROOT)
    if temperature_data is None:
        print("Jena dataset not found inside the root path.")
    else:
        mean_temp = temperature_data.mean()
        std_temp = temperature_data.std()
        data_norm = (temperature_data - mean_temp) / std_temp

        n = len(data_norm)
        train_data = data_norm[:int(0.7 * n)]
        val_data = data_norm[int(0.7 * n):int(0.9 * n)]
else:
    print("Jena dataset not found.")


def get_ts_loaders(seq_len=24, batch_size=64):
    if train_data is None or val_data is None:
        return None, None

    X_train, y_train = create_sequences(train_data, seq_len)
    X_val, y_val = create_sequences(val_data, seq_len)

    train_ds = TensorDataset(torch.tensor(X_train).unsqueeze(-1), torch.tensor(y_train))
    val_ds = TensorDataset(torch.tensor(X_val).unsqueeze(-1), torch.tensor(y_val))

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
    )


In [ ]:
class RNNModel(nn.Module):
    def __init__(
        self,
        input_size=1,
        hidden_size=64,
        num_layers=1,
        rnn_type="lstm",
        dropout=0.0,
        bidirectional=False,
    ):
        super().__init__()
        self.bidirectional = bidirectional
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(
                input_size,
                hidden_size,
                num_layers,
                batch_first=True,
                dropout=dropout,
                bidirectional=bidirectional,
            )
        elif rnn_type == "gru":
            self.rnn = nn.GRU(
                input_size,
                hidden_size,
                num_layers,
                batch_first=True,
                dropout=dropout,
                bidirectional=bidirectional,
            )
        else:
            self.rnn = nn.RNN(
                input_size,
                hidden_size,
                num_layers,
                batch_first=True,
                dropout=dropout,
                bidirectional=bidirectional,
            )

        out_dim = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(out_dim, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out).squeeze()


def train_ts(model, train_loader, val_loader, epochs=5, lr=1e-3):
    if train_loader is None or val_loader is None:
        return None

    model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_mse": [], "val_mse": [], "val_mae": []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_mse = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_mse += loss.item()

        train_mse /= max(1, len(train_loader))

        model.eval()
        val_mse = 0.0
        val_mae = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb)
                val_mse += criterion(preds, yb).item()
                val_mae += nn.L1Loss()(preds, yb).item()

        val_mse /= max(1, len(val_loader))
        val_mae /= max(1, len(val_loader))

        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)
        history["val_mae"].append(val_mae)

        print(
            f"Epoch {epoch} | Train MSE: {train_mse:.4f} | "
            f"Val MSE: {val_mse:.4f} | Val MAE: {val_mae:.4f}"
        )

    return history


def plot_ts_curves(history, title):
    if history is None:
        return
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_mse"], label="train_mse")
    plt.plot(history["val_mse"], label="val_mse")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.show()


In [ ]:
if JENA_ROOT.exists() and train_data is not None:
    RUN_RNN_EXPERIMENTS = True
    RNN_EXPERIMENT_LIMIT = None  # Set to an int for a quicker run
    PLOT_TS_CURVES = True
    PLOT_MAX = 6

    BASE_CONFIG = {
        "rnn_type": "lstm",
        "seq_len": 24,
        "hidden_size": 64,
        "num_layers": 1,
        "dropout": 0.0,
        "bidirectional": False,
        "batch_size": 64,
        "epochs": 3,
        "lr": 1e-3,
    }

    RNN_EXPERIMENTS = [
        {"name": "vanilla_rnn_seq24", "rnn_type": "rnn"},
        {"name": "lstm_seq24", "rnn_type": "lstm"},
        {"name": "gru_seq24", "rnn_type": "gru"},
        {"name": "lstm_seq72", "rnn_type": "lstm", "seq_len": 72},
        {"name": "lstm_hidden128", "rnn_type": "lstm", "hidden_size": 128},
        {"name": "lstm_2layers_dropout", "rnn_type": "lstm", "num_layers": 2, "dropout": 0.2},
        {"name": "lstm_bidirectional", "rnn_type": "lstm", "bidirectional": True},
    ]

    if RNN_EXPERIMENT_LIMIT is not None:
        RNN_EXPERIMENTS = RNN_EXPERIMENTS[:RNN_EXPERIMENT_LIMIT]

    plot_count = 0

    for cfg in RNN_EXPERIMENTS:
        exp_cfg = dict(BASE_CONFIG)
        exp_cfg.update(cfg)

        train_loader, val_loader = get_ts_loaders(
            seq_len=exp_cfg["seq_len"],
            batch_size=exp_cfg["batch_size"],
        )

        print(f"
[RNN] {exp_cfg['name']}")
        model = RNNModel(
            rnn_type=exp_cfg["rnn_type"],
            hidden_size=exp_cfg["hidden_size"],
            num_layers=exp_cfg["num_layers"],
            dropout=exp_cfg["dropout"],
            bidirectional=exp_cfg["bidirectional"],
        )

        history = train_ts(
            model,
            train_loader,
            val_loader,
            epochs=exp_cfg["epochs"],
            lr=exp_cfg["lr"],
        )

        if PLOT_TS_CURVES and plot_count < PLOT_MAX:
            plot_ts_curves(history, f"{exp_cfg['name']}")
            plot_count += 1


### Discussion Question (RNN)
* **Why are LSTMs and GRUs generally better than vanilla RNNs for long sequences?**
* **Explain the role of gates and how they help with vanishing gradients and long-term dependencies.**

*(Double-click to edit)*


## Wrap-up
- Summarize key findings.
- Optional: short error analysis.
